In [3]:
import os

In [4]:
import json
from pathlib import Path
from types import MethodType

import numpy as np
import torch
import torch.nn.functional as F
from vllm import LLM, SamplingParams


def find_repo_root():
    bases = [Path.cwd(), *Path.cwd().parents]
    candidates = [path for base in bases for path in (base, base / "Less_But_Sharper")]
    for path in candidates:
        if (path / "artifacts" / "masks").is_dir() and (path / "code" / "analysis").is_dir():
            return path.resolve()
    raise RuntimeError("Could not locate the Less_But_Sharper repository root.")


def first_existing(paths, label):
    for path in paths:
        if path.is_file():
            return path
    checked = "\n".join(f"  - {path}" for path in paths)
    raise FileNotFoundError(f"{label} was not found. Checked:\n{checked}")


REPO_ROOT = find_repo_root()
LEGACY_DATA_DIR = REPO_ROOT.parent / "data"
PACKAGE_DATA_DIR = REPO_ROOT / "data"
SIGNATURE_DIR = REPO_ROOT / "work" / "llama2-7b" / "signatures"
OUTPUT_DIR = REPO_ROOT / "work" / "analysis" / "causal"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LANGS = ["en", "zh", "fr", "es", "vi", "id", "ja"]
MODEL = "meta-llama/Llama-2-7b-hf"
MMR_PATH = REPO_ROOT / "artifacts" / "masks" / "llama2_mmr_k30_lambda0.6.pth"
EVAL_TOKENS = 2**18
OUT = OUTPUT_DIR / "causal_results.json"


def token_path(lang):
    return first_existing(
        [
            PACKAGE_DATA_DIR / f"id.{lang}.valid.llama2-7b",
            PACKAGE_DATA_DIR / f"id.{lang}.valid.llama",
            LEGACY_DATA_DIR / f"id.{lang}.valid.llama",
        ],
        f"Validation token IDs for {lang}",
    )


def signature_path(lang):
    return first_existing(
        [
            SIGNATURE_DIR / f"signature.{lang}.T30000.token.llama2-7b.pt",
            PACKAGE_DATA_DIR / f"signature.{lang}.T30000.token.llama2-7b.pt",
            PACKAGE_DATA_DIR / f"signature.{lang}.T30000.token.llama-7b.pt",
            LEGACY_DATA_DIR / f"signature.{lang}.T30000.token.llama-7b.pt",
        ],
        f"Token signature for {lang}",
    )


print(f"repo={REPO_ROOT}")
print(f"mask={MMR_PATH}")


/home/lar/miniconda3/envs/lang-neuron/lib/python3.10/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


repo=/home/lar/Language-Specific-Neurons/Less_But_Sharper
mask=/home/lar/Language-Specific-Neurons/Less_But_Sharper/artifacts/masks/llama2_mmr_k30_lambda0.6.pth


In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL)

/home/lar/miniconda3/envs/lang-neuron/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:

PAIRS = [
    ("zh", (12, 9050), (31, 9692), (18, 1684)),
    ("ja", (31, 6788), (30, 8956), (21, 3737)),
    ("vi", (14, 2367), (19, 6913), (28, 3693)),
]

In [7]:
model = LLM(model=MODEL, tensor_parallel_size=torch.cuda.device_count(),
            enforce_eager=True)
num_layers = model.llm_engine.model_config.hf_config.num_hidden_layers
max_length = model.llm_engine.model_config.max_model_len
print("layers:", num_layers, "max_len:", max_length)

INFO 11-25 07:05:37 llm_engine.py:70] Initializing an LLM engine with config: model='meta-llama/Llama-2-7b-hf', tokenizer='meta-llama/Llama-2-7b-hf', tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, quantization=None, enforce_eager=True, seed=0)
INFO 11-25 07:05:45 llm_engine.py:275] # GPU blocks: 3740, # CPU blocks: 512
layers: 32 max_len: 4096


In [8]:
def coords_to_mask(coords):
    per_layer = [[] for _ in range(num_layers)]
    for l, j in coords:
        per_layer[l].append(j)
    return [torch.tensor(sorted(v), dtype=torch.long) for v in per_layer]


def load_mmr_coords(lang):
    data = torch.load(MMR_PATH, map_location="cpu")
    per_layer = data[LANGS.index(lang)]
    return [(l, int(j)) for l, idxs in enumerate(per_layer) for j in idxs.tolist()]


def apply_mask(mask_per_layer):
    """Apply a per-layer mask; use None to clear it."""
    for i in range(num_layers):
        layer_mask = (mask_per_layer[i] if mask_per_layer is not None
                      else torch.tensor([], dtype=torch.long)).to("cuda")

        def factory(mask):
            def llama_forward(self, x):
                gate_up, _ = self.gate_up_proj(x)
                half = gate_up.size(-1) // 2
                act = F.silu(gate_up[:, :, :half])
                if mask.numel() > 0:
                    act.index_fill_(2, mask, 0)
                x = act * gate_up[:, :, half:]
                x, _ = self.down_proj(x)
                return x
            return llama_forward

        obj = model.llm_engine.driver_worker.model_runner.model.model.layers[i].mlp
        obj.forward = MethodType(factory(layer_mask), obj)

In [9]:
def _lp(v):
    """Read a float or vLLM Logprob value."""
    return v if isinstance(v, (int, float)) else v.logprob


_input_cache = {}

def eval_ppl(lang):
    if lang not in _input_cache:
        ids = torch.load(token_path(lang), map_location="cpu")
        l = min(ids.size(0), EVAL_TOKENS) // max_length * max_length
        _input_cache[lang] = ids[:l].reshape(-1, max_length).tolist()

    outputs = model.generate(
        prompt_token_ids=_input_cache[lang],
        sampling_params=SamplingParams(max_tokens=1, prompt_logprobs=0),
    )
    vals = []
    for o in outputs:
        row = [_lp(next(iter(r.values()))) for r in o.prompt_logprobs if r]
        vals.append(np.mean(row))
    return float(-np.mean(vals))

In [10]:
lang, d, s, s_ctrl = PAIRS[0]
S_coords = load_mmr_coords(lang)

apply_mask(None)
base = eval_ppl(lang)
print(f"[{lang}] baseline PPL = {base:.4f}")

for name, coords in [("ablate_d", [d]), ("ablate_s", [s]),
                     ("ablate_d+s", [d, s]), ("ablate_d+sctrl", [d, s_ctrl])]:
    apply_mask(coords_to_mask(coords))
    ppl = eval_ppl(lang)
    apply_mask(None)
    print(f"[{lang}] {name:<16} PPL={ppl:.4f}  ΔPPL={ppl-base:+.5f}")

Processed prompts: 100%|██████████| 64/64 [01:38<00:00,  1.54s/it]


[zh] baseline PPL = 1.5711


Processed prompts: 100%|██████████| 64/64 [01:32<00:00,  1.45s/it]


[zh] ablate_d         PPL=1.5713  ΔPPL=+0.00023


Processed prompts: 100%|██████████| 64/64 [01:35<00:00,  1.49s/it]


[zh] ablate_s         PPL=1.5803  ΔPPL=+0.00922


Processed prompts: 100%|██████████| 64/64 [01:33<00:00,  1.46s/it]


[zh] ablate_d+s       PPL=1.5806  ΔPPL=+0.00946


Processed prompts: 100%|██████████| 64/64 [01:35<00:00,  1.50s/it]

[zh] ablate_d+sctrl   PPL=1.5713  ΔPPL=+0.00022


In [11]:
results = []
baseline_cache = {}

for lang, d, s, s_ctrl in PAIRS:
    S_coords = load_mmr_coords(lang)
    S_set = set(S_coords)
    conditions = {
        "baseline":          None,
        "ablate_S":          S_coords,
        "ablate_S-s":        sorted(S_set - {s}),
        "ablate_S-s+d":      sorted((S_set - {s}) | {d}),
        "ablate_S-s+sctrl":  sorted((S_set - {s}) | {s_ctrl}),
        "ablate_S+d":        S_coords + [d],
    }
    for name, coords in conditions.items():
        if name == "baseline" and lang in baseline_cache:
            ppl = baseline_cache[lang]
        else:
            apply_mask(None if coords is None else coords_to_mask(coords))
            ppl = eval_ppl(lang)
            apply_mask(None)
            if name == "baseline":
                baseline_cache[lang] = ppl
        base = baseline_cache.get(lang)
        dppl = round(ppl - base, 5) if (base is not None and name != "baseline") else 0.0
        print(f"[{lang}] {name:<16} PPL={ppl:.4f}  ΔPPL={dppl:+.5f}")
        results.append({"lang": lang, "cond": name,
                        "pair": {"d": list(d), "s": list(s), "s_ctrl": list(s_ctrl)},
                        "ppl": round(ppl, 5), "dppl": dppl})

with open(OUT, "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved:", OUT)

Processed prompts: 100%|██████████| 64/64 [01:33<00:00,  1.46s/it]


[zh] baseline         PPL=1.5711  ΔPPL=+0.00000


Processed prompts: 100%|██████████| 64/64 [01:36<00:00,  1.50s/it]


[zh] ablate_S         PPL=1.5945  ΔPPL=+0.02337


Processed prompts: 100%|██████████| 64/64 [01:32<00:00,  1.45s/it]


[zh] ablate_S-s       PPL=1.5811  ΔPPL=+0.01004


... (26 outputs left out)

Processed prompts: 100%|██████████| 64/64 [01:31<00:00,  1.43s/it]


[vi] ablate_S-s+sctrl PPL=1.0210  ΔPPL=+0.00761


Processed prompts: 100%|██████████| 64/64 [01:34<00:00,  1.48s/it]

[vi] ablate_S+d       PPL=1.0423  ΔPPL=+0.02887

Saved: /home/lar/Language-Specific-Neurons/Less_But_Sharper/work/analysis/causal/causal_results.json


In [12]:
from collections import defaultdict
by = defaultdict(dict)
for r in results:
    by[r["lang"]][r["cond"]] = r["dppl"]

print(f"{'lang':<5}{'Δd':>9}{'Δs':>9}{'Δd+s':>9}{'sub-add':>9} | "
      f"{'Δd+sctrl':>10}{'sub-add(ctrl)':>15} | {'ΔS':>9}{'ΔS+d':>9}")
print("-" * 92)
for lang in by:
    b = by[lang]
    sub      = b.get("ablate_d",0)+b.get("ablate_s",0)-b.get("ablate_d+s",0)
    sub_ctrl = b.get("ablate_d",0)+b.get("ablate_s",0)-b.get("ablate_d+sctrl",0)
    print(f"{lang:<5}{b.get('ablate_d',0):>9.4f}{b.get('ablate_s',0):>9.4f}"
          f"{b.get('ablate_d+s',0):>9.4f}{sub:>9.4f} | "
          f"{b.get('ablate_d+sctrl',0):>10.4f}{sub_ctrl:>15.4f} | "
          f"{b.get('ablate_S',0):>9.4f}{b.get('ablate_S+d',0):>9.4f}")

print("\nPositive sub-additivity with a near-zero control indicates redundancy.")
print("      If ΔS+d ≈ ΔS, the selected set already covers d.")

lang        Δd       Δs     Δd+s  sub-add |   Δd+sctrl  sub-add(ctrl) |        ΔS     ΔS+d
--------------------------------------------------------------------------------------------
zh      0.0000   0.0000   0.0000   0.0000 |     0.0000         0.0000 |    0.0234   0.0236
ja      0.0000   0.0000   0.0000   0.0000 |     0.0000         0.0000 |    0.0202   0.0204
vi      0.0000   0.0000   0.0000   0.0000 |     0.0000         0.0000 |    0.0282   0.0289

Positive sub-additivity with a near-zero control indicates redundancy.
      If ΔS+d ≈ ΔS, the selected set already covers d.


In [13]:
for lang, d, s, s_ctrl in PAIRS:
    S = set(load_mmr_coords(lang))
    print(f"{lang}: s in S? {s in S} | s' in S? {s_ctrl in S} | d in S? {d in S}")

zh: s in S? True | s' in S? True | d in S? False
ja: s in S? True | s' in S? True | d in S? False
vi: s in S? True | s' in S? True | d in S? False


In [14]:

MIN_CORPUS_COUNT = 30

def sig_path(lang):
    return signature_path(lang)

def load_signature(path):
    data = torch.load(path, map_location="cpu")
    coords = [tuple(c) for c in data["coords"]]
    sig = data["signature"]
    row_of = {c: i for i, c in enumerate(coords)}
    return coords, sig, row_of, int(data.get("num_tokens", sig.shape[1]))

def jaccard_bool(a, b):
    inter = (a & b).sum().item()
    union = (a | b).sum().item()
    return inter / union if union > 0 else 0.0

In [15]:


def pick_sctrl(lang, d, S_set, row_of, sig, coords):
    """Return candidates outside S with the lowest Jaccard similarity to d."""
    rd = row_of[d]
    cands = []
    for c, r in row_of.items():
        if c in S_set or c == d:
            continue
        j = jaccard_bool(sig[rd], sig[r])
        pa = sig[r].float().mean().item()

        if 0.3 < pa < 0.95:
            cands.append((j, c, pa))
    cands.sort()
    return cands[:5]


NEW_PAIRS = []
for lang, d, s, _ in PAIRS:
    S_set = set(load_mmr_coords(lang))
    coords, sig, row_of, num_tokens = load_signature(sig_path(lang))
    cands = pick_sctrl(lang, d, S_set, row_of, sig, coords)
    j_ds = jaccard_bool(sig[row_of[d]], sig[row_of[s]])
    print(f"\n[{lang}] d={d}  s={s}  (J={j_ds:.3f})")
    print("  Control candidates outside S, ordered by Jaccard to d:")
    for j, c, pa in cands:
        print(f"    {c}  J(d,·)={j:.3f}  activation_rate={pa:.3f}")
    NEW_PAIRS.append((lang, d, s, cands[0][1]))

print("\nSelected pairs:", NEW_PAIRS)


[zh] d=(12, 9050)  s=(31, 9692)  (J=0.980)
  Control candidates outside S, ordered by Jaccard to d:
    (21, 3390)  J(d,·)=0.402  activation_rate=0.406
    (0, 1578)  J(d,·)=0.403  activation_rate=0.397
    (21, 6355)  J(d,·)=0.445  activation_rate=0.446
    (0, 8017)  J(d,·)=0.496  activation_rate=0.505
    (20, 2824)  J(d,·)=0.503  activation_rate=0.504

[ja] d=(31, 6788)  s=(30, 8956)  (J=0.922)
  Control candidates outside S, ordered by Jaccard to d:
    (15, 10982)  J(d,·)=0.469  activation_rate=0.530
    (0, 9091)  J(d,·)=0.481  activation_rate=0.526
    (23, 2157)  J(d,·)=0.486  activation_rate=0.503
    (30, 2484)  J(d,·)=0.489  activation_rate=0.517
    (20, 6616)  J(d,·)=0.489  activation_rate=0.532

[vi] d=(14, 2367)  s=(19, 6913)  (J=0.970)
  Control candidates outside S, ordered by Jaccard to d:
    (24, 8767)  J(d,·)=0.551  activation_rate=0.557
    (26, 7211)  J(d,·)=0.565  activation_rate=0.566
    (27, 10492)  J(d,·)=0.580  activation_rate=0.581
    (0, 5344)  J(d,·)=

In [16]:
results2 = []
baseline_cache = {}

for lang, d, s, s_ctrl in NEW_PAIRS:
    S_coords = load_mmr_coords(lang)
    S_set = set(S_coords)

    conditions = {
        "baseline":          None,
        "ablate_S":          S_coords,
        "ablate_S-s":        sorted(S_set - {s}),
        "ablate_S-s+d":      sorted((S_set - {s}) | {d}),
        "ablate_S-s+sctrl":  sorted((S_set - {s}) | {s_ctrl}),
    }
    for name, cs in conditions.items():
        if name == "baseline" and lang in baseline_cache:
            ppl = baseline_cache[lang]
        else:
            apply_mask(None if cs is None else coords_to_mask(cs))
            ppl = eval_ppl(lang)
            apply_mask(None)
            if name == "baseline":
                baseline_cache[lang] = ppl
        base = baseline_cache[lang]
        dppl = round(ppl - base, 5) if name != "baseline" else 0.0
        n = 0 if cs is None else len(cs)
        print(f"[{lang}] {name:<18} (n={n:2d})  PPL={ppl:.4f}  ΔPPL={dppl:+.5f}")
        results2.append({"lang": lang, "cond": name, "n": n, "ppl": round(ppl,5), "dppl": dppl})


print(f"\n{'lang':<5}{'ΔS':>9}{'ΔS-s':>9}{'ΔS-s+d':>10}{'ΔS-s+s′':>10}")
print("-" * 44)
from collections import defaultdict
by = defaultdict(dict)
for r in results2: by[r["lang"]][r["cond"]] = r["dppl"]
for lang in by:
    b = by[lang]
    print(f"{lang:<5}{b.get('ablate_S',0):>9.4f}{b.get('ablate_S-s',0):>9.4f}"
          f"{b.get('ablate_S-s+d',0):>10.4f}{b.get('ablate_S-s+sctrl',0):>10.4f}")
print("\nReplacement succeeds when d restores the effect and the control does not.")

Processed prompts: 100%|██████████| 64/64 [01:31<00:00,  1.44s/it]


[zh] baseline           (n= 0)  PPL=1.5711  ΔPPL=+0.00000


Processed prompts: 100%|██████████| 64/64 [01:34<00:00,  1.47s/it]


[zh] ablate_S           (n=30)  PPL=1.5945  ΔPPL=+0.02337


Processed prompts: 100%|██████████| 64/64 [01:32<00:00,  1.44s/it]


[zh] ablate_S-s         (n=29)  PPL=1.5811  ΔPPL=+0.01004


... (20 outputs left out)

Processed prompts: 100%|██████████| 64/64 [01:27<00:00,  1.36s/it]


[vi] ablate_S-s+d       (n=30)  PPL=1.0219  ΔPPL=+0.00847


Processed prompts: 100%|██████████| 64/64 [01:27<00:00,  1.36s/it]

[vi] ablate_S-s+sctrl   (n=30)  PPL=1.0210  ΔPPL=+0.00764

lang        ΔS     ΔS-s    ΔS-s+d   ΔS-s+s′
--------------------------------------------
zh      0.0234   0.0100    0.0103    0.0100
ja      0.0202   0.0181    0.0182    0.0181
vi      0.0282   0.0076    0.0085    0.0076

Replacement succeeds when d restores the effect and the control does not.


In [17]:
PAIRS = [
    ("ja", (16, 1048), (16, 1532), (21, 3737)),
]

In [18]:
def eval_ppl_by_script(lang):
    if lang not in _input_cache:
        ids_t = torch.load(token_path(lang), map_location="cpu")
        length = min(ids_t.size(0), EVAL_TOKENS)
        length = length // max_length * max_length
        _input_cache[lang] = ids_t[:length].reshape(-1, max_length).tolist()

    prompts = _input_cache[lang]
    outputs = model.generate(
        prompt_token_ids=prompts,
        sampling_params=SamplingParams(
            max_tokens=1,
            prompt_logprobs=0,
        ),
    )

    buckets = {
        "Kanji": [],
        "Hiragana": [],
        "Katakana": [],
        "byte_piece": [],
        "other": [],
    }

    for prompt, output in zip(prompts, outputs):
        for token_id, result in zip(prompt, output.prompt_logprobs):
            if not result:
                continue

            logprob = _lp(next(iter(result.values())))
            piece = tokenizer.convert_ids_to_tokens(token_id)
            text = piece.replace("\u2581", "")

            if piece.startswith("<0x"):
                category = "byte_piece"
            elif not text:
                category = "other"
            elif "\u4e00" <= text[0] <= "\u9fff":
                category = "Kanji"
            elif "\u3040" <= text[0] <= "\u309f":
                category = "Hiragana"
            elif "\u30a0" <= text[0] <= "\u30ff":
                category = "Katakana"
            else:
                category = "other"

            buckets[category].append(logprob)

    result = {
        category: float(-np.mean(values)) if values else float("nan")
        for category, values in buckets.items()
    }

    all_logprobs = [
        value
        for values in buckets.values()
        for value in values
    ]
    result["overall"] = float(-np.mean(all_logprobs))
    result["_counts"] = {
        category: len(values)
        for category, values in buckets.items()
    }
    return result

In [19]:
lang, d, s, s_ctrl = PAIRS[0]
CATS = ["Kanji", "Hiragana", "Katakana", "overall"]

conditions = {
    "baseline":   None,
    "ablate_d":   [d],
    "ablate_s":   [s],
    "ablate_s'":  [s_ctrl],
    "ablate_d+s": [d, s],
}

res = {}
for name, cs in conditions.items():
    apply_mask(None if cs is None else coords_to_mask(cs))
    res[name] = eval_ppl_by_script(lang)
    apply_mask(None)
    print(f"[{name}] done")

base = res["baseline"]
print(f"\n{'condition':<12}" + "".join(f"{c:>11}" for c in CATS))
print("-" * 58)
for name in conditions:
    if name == "baseline":
        print(f"{'baseline':<12}" + "".join(f"{base[c]:>11.4f}" for c in CATS))
        continue
    print(f"{name:<12}" + "".join(f"{res[name][c]-base[c]:>+11.5f}" for c in CATS))

print("\nCompare script-specific PPL changes for d and s.")
print("      The control should affect a different script axis.")

Processed prompts: 100%|██████████| 64/64 [01:27<00:00,  1.36s/it]


[baseline] done


Processed prompts: 100%|██████████| 64/64 [01:27<00:00,  1.36s/it]


[ablate_d] done


Processed prompts: 100%|██████████| 64/64 [01:27<00:00,  1.37s/it]


[ablate_s] done


Processed prompts: 100%|██████████| 64/64 [01:27<00:00,  1.37s/it]


[ablate_s'] done


Processed prompts: 100%|██████████| 64/64 [01:32<00:00,  1.44s/it]


[ablate_d+s] done

condition         Kanji   Hiragana   Katakana    overall
----------------------------------------------------------
baseline         1.9132     1.1402     1.1819     1.2262
ablate_d       +0.00005   +0.00070   -0.00007   +0.00010
ablate_s       +0.00170   -0.00017   -0.00104   +0.00027
ablate_s'      -0.00012   -0.00011   +0.00158   +0.00020
ablate_d+s     +0.00172   +0.00055   -0.00113   +0.00039

Compare script-specific PPL changes for d and s.
      The control should affect a different script axis.


In [20]:

lang = "ja"
ids_t = torch.load(token_path(lang), map_location="cpu")
l = min(ids_t.size(0), EVAL_TOKENS) // max_length * max_length
sample = ids_t[:l].reshape(-1, max_length)[0].tolist()[:200]

from collections import Counter
cnt = Counter()
for tid in sample:
    p = tokenizer.convert_ids_to_tokens(tid)
    s = p.replace("\u2581", "")
    if not s: cnt["empty"] += 1; continue
    c = s[0]
    if   "\u4e00" <= c <= "\u9fff": cnt["Kanji"] += 1
    elif "\u3040" <= c <= "\u309f": cnt["Hiragana"] += 1
    elif "\u30a0" <= c <= "\u30ff": cnt["Katakana"] += 1
    elif p.startswith("<0x"):       cnt["byte_piece"] += 1
    else:                           cnt["other"] += 1
print(cnt)
print("\nToken sample:", [tokenizer.convert_ids_to_tokens(t) for t in sample[:30]])

Counter({'Kanji': 75, 'byte_piece': 48, 'Hiragana': 38, 'other': 36, 'empty': 3})

Token sample: ['▁', '御', '<0xE6>', '<0xAE>', '<0xBF>', '場', '線', '（', 'ご', 'て', 'ん', 'ば', 'せ', 'ん', '）', 'は', '、', '神', '奈', '川', '県', '小', '田', '原', '市', 'の', '国', '府', '津', '駅']


In [21]:
print(type(base), base)

<class 'dict'> {'Kanji': 1.9132164549741515, 'Hiragana': 1.1401599261832758, 'Katakana': 1.181874812027823, 'byte_piece': 0.8283629307885951, 'other': 1.1962303288406524, 'overall': 1.2261631574754677, '_counts': {'Kanji': 50873, 'Hiragana': 64004, 'Katakana': 39890, 'byte_piece': 66516, 'other': 40797}}


In [22]:
lang = "ja"
prompts = _input_cache[lang]
prompt = prompts[0]
pieces = tokenizer.convert_ids_to_tokens(prompt)

from collections import Counter
cnt = Counter()
hira_samples = []
for pos in range(min(300, len(pieces))):
    p = pieces[pos]
    s = p.replace("\u2581", "")
    if not s:
        cnt["empty"] += 1; continue
    c = s[0]
    if   "\u4e00" <= c <= "\u9fff": cnt["Kanji"] += 1
    elif "\u3040" <= c <= "\u309f":
        cnt["Hiragana"] += 1
        hira_samples.append((pos, p, repr(c), hex(ord(c))))
    elif "\u30a0" <= c <= "\u30ff": cnt["Katakana"] += 1
    elif p.startswith("<0x"):       cnt["byte_piece"] += 1
    else:                           cnt["other"] += 1

print(cnt)
print("Hiragana sample:", hira_samples[:5])
print("\nprompt_logprobs length:", len(_last_outputs[0].prompt_logprobs) if '_last_outputs' in dir() else "N/A")
print("prompt length:", len(prompt))

Counter({'Kanji': 113, 'byte_piece': 73, 'Hiragana': 60, 'other': 41, 'Katakana': 8, 'empty': 5})
Hiragana sample: [(8, 'ご', "'ご'", '0x3054'), (9, 'て', "'て'", '0x3066'), (10, 'ん', "'ん'", '0x3093'), (11, 'ば', "'ば'", '0x3070'), (12, 'せ', "'せ'", '0x305b')]

prompt_logprobs length: N/A
prompt length: 4096


In [23]:
lang = "ja"
ids_t = torch.load(token_path(lang), map_location="cpu")
l = min(ids_t.size(0), EVAL_TOKENS) // max_length * max_length
prompts = ids_t[:l].reshape(-1, max_length).tolist()
prompt = prompts[0]

from collections import Counter
cnt = Counter()
hira_samples = []

for pos in range(min(300, len(prompt))):
    p = tokenizer.convert_ids_to_tokens(prompt[pos])
    s = p.replace("\u2581", "")
    if not s:
        cnt["empty"] += 1; continue
    c = s[0]
    if   "\u4e00" <= c <= "\u9fff": cnt["Kanji"] += 1
    elif "\u3040" <= c <= "\u309f":
        cnt["Hiragana"] += 1
        hira_samples.append((pos, p))
    elif "\u30a0" <= c <= "\u30ff": cnt["Katakana"] += 1
    elif p.startswith("<0x"):       cnt["byte_piece"] += 1
    else:                           cnt["other"] += 1

print(cnt)
print("Hiragana sample:", hira_samples[:8])


pieces_list = tokenizer.convert_ids_to_tokens(prompt[:300])
pieces_scalar = [tokenizer.convert_ids_to_tokens(t) for t in prompt[:300]]
print("\nList and scalar token conversion match:", pieces_list == pieces_scalar)
print("length:", len(pieces_list), len(pieces_scalar), "prompt:", len(prompt[:300]))

Counter({'Kanji': 113, 'byte_piece': 73, 'Hiragana': 60, 'other': 41, 'Katakana': 8, 'empty': 5})
Hiragana sample: [(8, 'ご'), (9, 'て'), (10, 'ん'), (11, 'ば'), (12, 'せ'), (13, 'ん'), (15, 'は'), (25, 'の')]

List and scalar token conversion match: True
length: 300 300 prompt: 300


In [24]:
def eval_ppl_by_script(lang):
    if lang not in _input_cache:
        ids_t = torch.load(token_path(lang), map_location="cpu")
        l = min(ids_t.size(0), EVAL_TOKENS) // max_length * max_length
        _input_cache[lang] = ids_t[:l].reshape(-1, max_length).tolist()

    prompts = _input_cache[lang]
    outputs = model.generate(prompt_token_ids=prompts,
                             sampling_params=SamplingParams(max_tokens=1, prompt_logprobs=0))

    buckets = {"Kanji": [], "Hiragana": [], "Katakana": [], "byte_piece": [], "other": []}

    for prompt, o in zip(prompts, outputs):
        lps = o.prompt_logprobs
        assert len(lps) == len(prompt), f"Length mismatch: {len(lps)} vs {len(prompt)}"
        for pos in range(len(prompt)):
            r = lps[pos]
            if not r:
                continue
            lp = _lp(next(iter(r.values())))
            p = tokenizer.convert_ids_to_tokens(prompt[pos])
            s = p.replace("\u2581", "")
            if not s:
                buckets["other"].append(lp); continue
            c = s[0]
            if   "\u4e00" <= c <= "\u9fff": buckets["Kanji"].append(lp)
            elif "\u3040" <= c <= "\u309f": buckets["Hiragana"].append(lp)
            elif "\u30a0" <= c <= "\u30ff": buckets["Katakana"].append(lp)
            elif p.startswith("<0x"):       buckets["byte_piece"].append(lp)
            else:                           buckets["other"].append(lp)

    out = {k: (float(-np.mean(v)) if v else None) for k, v in buckets.items()}
    out["overall"] = float(-np.mean([x for v in buckets.values() for x in v]))
    out["_counts"] = {k: len(v) for k, v in buckets.items()}
    return out

In [25]:
apply_mask(None)
b = eval_ppl_by_script("ja")
print(b["_counts"])
print({k: v for k, v in b.items() if k != "_counts"})

Processed prompts: 100%|██████████| 64/64 [01:29<00:00,  1.39s/it]


{'Kanji': 50873, 'Hiragana': 64004, 'Katakana': 39890, 'byte_piece': 66516, 'other': 40797}
{'Kanji': 1.9132164549741515, 'Hiragana': 1.1401599261832758, 'Katakana': 1.181874812027823, 'byte_piece': 0.8283629307885951, 'other': 1.1962303288406524, 'overall': 1.2261631574754677}


In [26]:
lang, d, s, s_ctrl = ("ja", (16, 1048), (16, 1532), (21, 3737))
CATS = ["Kanji", "Hiragana", "Katakana", "byte_piece", "overall"]

conditions = {
    "baseline":   None,
    "ablate_d":   [d],
    "ablate_s":   [s],
    "ablate_sctrl": [s_ctrl],
    "ablate_d+s": [d, s],
}

res = {}
for name, cs in conditions.items():
    apply_mask(None if cs is None else coords_to_mask(cs))
    res[name] = eval_ppl_by_script(lang)
    apply_mask(None)
    print(f"[{name}] done")

base = res["baseline"]
print(f"\n{'condition':<14}" + "".join(f"{c:>11}" for c in CATS))
print("-" * (14 + 11 * len(CATS)))
print(f"{'baseline':<14}" + "".join(f"{base[c]:>11.4f}" for c in CATS))
for name in conditions:
    if name == "baseline":
        continue
    print(f"{name:<14}" + "".join(f"{res[name][c] - base[c]:>+11.5f}" for c in CATS))

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 64/64 [01:30<00:00,  1.42s/it]


[baseline] done


Processed prompts: 100%|██████████| 64/64 [01:29<00:00,  1.40s/it]


[ablate_d] done


Processed prompts: 100%|██████████| 64/64 [01:31<00:00,  1.44s/it]


[ablate_s] done


Processed prompts: 100%|██████████| 64/64 [01:30<00:00,  1.41s/it]


[ablate_sctrl] done


Processed prompts: 100%|██████████| 64/64 [01:31<00:00,  1.43s/it]


[ablate_d+s] done

condition           Kanji   Hiragana   Katakana byte_piece    overall
---------------------------------------------------------------------
baseline           1.9132     1.1402     1.1819     0.8284     1.2262
ablate_d         +0.00005   +0.00070   -0.00007   -0.00005   +0.00010
ablate_s         +0.00170   -0.00017   -0.00104   +0.00074   +0.00027
ablate_sctrl     -0.00012   -0.00011   +0.00158   +0.00021   +0.00020
ablate_d+s       +0.00172   +0.00055   -0.00113   +0.00070   +0.00039


In [27]:
lang = "ja"
CATS = ["Kanji", "Hiragana", "Katakana", "byte_piece", "overall"]


D_GRP  = [(16, 1048), (15, 9463)]
S_GRP  = [(16, 1532), (19, 6891), (10, 4637)]
SC_GRP = [(21, 3737), (26, 10653), (31, 10895)]

conditions = {
    "baseline":        None,
    "ablate_D":        D_GRP,
    "ablate_S":        S_GRP,
    "ablate_Sctrl":    SC_GRP,
    "ablate_D+S":      D_GRP + S_GRP,
    "ablate_D+Sctrl":  D_GRP + SC_GRP,
}

res = {}
for name, cs in conditions.items():
    apply_mask(None if cs is None else coords_to_mask(cs))
    res[name] = eval_ppl_by_script(lang)
    apply_mask(None)
    print(f"[{name}] done (n={0 if cs is None else len(cs)})")

base = res["baseline"]
print(f"\n{'condition':<16}" + "".join(f"{c:>11}" for c in CATS))
print("-" * (16 + 11 * len(CATS)))
print(f"{'baseline':<16}" + "".join(f"{base[c]:>11.4f}" for c in CATS))
for name in conditions:
    if name == "baseline":
        continue
    print(f"{name:<16}" + "".join(f"{res[name][c] - base[c]:>+11.5f}" for c in CATS))


print("\n=== Kanji/Hiragana additivity ===")
for cat in ["Kanji", "Hiragana"]:
    dD  = res["ablate_D"][cat] - base[cat]
    dS  = res["ablate_S"][cat] - base[cat]
    dDS = res["ablate_D+S"][cat] - base[cat]
    print(f"{cat}: ΔD={dD:+.5f}  ΔS={dS:+.5f}  sum={dD+dS:+.5f}  ΔD+S={dDS:+.5f}  "
          f"sub-add={dD+dS-dDS:+.5f}")

Processed prompts: 100%|██████████| 64/64 [01:30<00:00,  1.41s/it]


[baseline] done (n=0)


Processed prompts: 100%|██████████| 64/64 [01:30<00:00,  1.41s/it]


[ablate_D] done (n=2)


Processed prompts: 100%|██████████| 64/64 [01:31<00:00,  1.43s/it]


[ablate_S] done (n=3)


... (2 outputs left out)

Processed prompts: 100%|██████████| 64/64 [01:27<00:00,  1.36s/it]


[ablate_D+S] done (n=5)


Processed prompts: 100%|██████████| 64/64 [01:27<00:00,  1.36s/it]


[ablate_D+Sctrl] done (n=5)

condition             Kanji   Hiragana   Katakana byte_piece    overall
-----------------------------------------------------------------------
baseline             1.9132     1.1402     1.1819     0.8284     1.2262
ablate_D           +0.00025   +0.00095   -0.00017   -0.00005   +0.00013
ablate_S           +0.00209   -0.00002   -0.00099   +0.00085   +0.00046
ablate_Sctrl       +0.00532   -0.00145   +0.00401   -0.00031   +0.00043
ablate_D+S         +0.00229   +0.00096   -0.00116   +0.00078   +0.00060
ablate_D+Sctrl     +0.00551   -0.00049   +0.00384   -0.00034   +0.00057

=== Kanji/Hiragana additivity ===
Kanji: ΔD=+0.00025  ΔS=+0.00209  sum=+0.00234  ΔD+S=+0.00229  sub-add=+0.00005
Hiragana: ΔD=+0.00095  ΔS=-0.00002  sum=+0.00092  ΔD+S=+0.00096  sub-add=-0.00004


In [28]:
lang = "ja"
coords, sig, row_of, num_tokens = load_signature(sig_path(lang))

d, s, s_ctrl = (16, 1048), (16, 1532), (21, 3737)

print(f"{'pair':<32}{'Jaccard':>9}{'p_A':>7}{'p_B':>7}")
print("-" * 55)
for label, (a, b) in [("d vs s (same axis)", (d, s)),
                      ("d vs s' (different axis)", (d, s_ctrl)),
                      ("s vs s' (both selected)", (s, s_ctrl))]:
    A, B = sig[row_of[a]], sig[row_of[b]]
    j = jaccard_bool(A, B)
    pa, pb = A.float().mean().item(), B.float().mean().item()
    print(f"{label + ' ' + str(a) + '↔' + str(b):<32}{j:>9.4f}{pa:>7.3f}{pb:>7.3f}")

pair                              Jaccard    p_A    p_B
-------------------------------------------------------
d vs s (same axis) (16, 1048)↔(16, 1532)   0.7564  0.914  0.738
d vs s' (different axis) (16, 1048)↔(21, 3737)   0.7199  0.914  0.770
s vs s' (both selected) (16, 1532)↔(21, 3737)   0.5864  0.738  0.770
